# GSA for Robust XAI — SAMO Summer School 2026
**Giulia Vannucci · University of Naples Federico II**

We apply Global Sensitivity Analysis (GSA) with Sobol indices to a real clinical black-box model: a DenseNet-121 CNN trained for melanoma classification.

- **Y = P(melanoma)** — predicted probability
- **X** = 5 photometric parameters: brightness, contrast, sharpness, saturation, hue
- The model is a **fixed black box** — no internal access required

> Vannucci, Coppolecchia & Siciliano (2026). *Global Sensitivity Analysis for Robust XAI*. **Risk Analysis**.

In [3]:
!pip install salib opencv-python tqdm gdown -q

import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import os, time
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageEnhance
from io import BytesIO
import requests
import gdown
from SALib.sample import sobol as sobol_sample
from SALib.analyze import sobol as sobol_analyze
from tensorflow.keras.applications.densenet import preprocess_input

print('TF version :', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

TF version : 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Load model and images

In [5]:
# ── Results directory (local on Colab) ─────────────────
RESULTS_PATH = '/content/results'
os.makedirs(RESULTS_PATH, exist_ok=True)
print(f'Results will be saved to: {RESULTS_PATH}')


# ── Model from Google Drive ────────────────────────────
MODEL_FILE_ID = '15oIhhhjw4UrTv7ZuZE6wldkF_eVRMhr_'
MODEL_PATH = 'melanoma_model.keras'

if not os.path.exists(MODEL_PATH):
    print('Downloading model...')
    gdown.download(
        f'https://drive.google.com/uc?id={MODEL_FILE_ID}',
        MODEL_PATH,
        quiet=False
    )
else:
    print('Model already present.')

melanoma_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'preprocess_input': preprocess_input},
    compile=False
)

print('Model loaded. Input shape:', melanoma_model.input_shape)


# ── Images from GitHub ─────────────────────────────────
REPO_OWNER = 'Giuliana87'
REPO_NAME = 'Samo2026'
IMG_SIZE = (128, 128)

# Label corretti dalla schermata:
# 0 = Non-Melanoma
# 1 = Melanoma
IMAGE_INFO = [
    {'name': 'ISIC_0024452', 'filename': 'ISIC_0024452.jpg', 'label': 0},
    {'name': 'ISIC_0024658', 'filename': 'ISIC_0024658.jpg', 'label': 0},
    {'name': 'ISIC_0024674', 'filename': 'ISIC_0024674.jpg', 'label': 0},
    {'name': 'ISIC_0024710', 'filename': 'ISIC_0024710.jpg', 'label': 0},
    {'name': 'ISIC_0024855', 'filename': 'ISIC_0024855.jpg', 'label': 0},
    {'name': 'ISIC_0024940', 'filename': 'ISIC_0024940.jpg', 'label': 1},
    {'name': 'ISIC_0025081', 'filename': 'ISIC_0025081.jpg', 'label': 1},
    {'name': 'ISIC_0025248', 'filename': 'ISIC_0025248.jpg', 'label': 1},
    {'name': 'ISIC_0025472', 'filename': 'ISIC_0025472.jpg', 'label': 1},
    {'name': 'ISIC_0025520', 'filename': 'ISIC_0025520.jpg', 'label': 1},
]


def get_repo_tree(owner, repo):
    for branch in ['main', 'master']:
        api_url = f'https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1'
        response = requests.get(api_url)

        if response.status_code == 200:
            print(f'GitHub branch found: {branch}')
            return response.json()['tree'], branch

    raise ValueError('Non riesco a leggere il repository GitHub. Controlla che sia pubblico e che il nome sia corretto.')


def find_image_url(filename, tree, branch):
    for item in tree:
        path = item.get('path', '')

        if path.endswith(filename):
            return f'https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/{branch}/{path}'

    raise ValueError(f'Immagine non trovata nel repository: {filename}')


def load_image_from_url(url, size=IMG_SIZE):
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError(f"Errore nel caricamento dell'immagine: {url}")

    img = Image.open(BytesIO(response.content)).convert('RGB')
    return img.resize(size, Image.LANCZOS)


print('Reading GitHub repository...')
repo_tree, branch = get_repo_tree(REPO_OWNER, REPO_NAME)

IMAGE_LIST = []

for info in IMAGE_INFO:
    url = find_image_url(info['filename'], repo_tree, branch)

    IMAGE_LIST.append({
        'name': info['name'],
        'filename': info['filename'],
        'url': url,
        'label': info['label']
    })

print('Loading images...')

for item in IMAGE_LIST:
    item['pil'] = load_image_from_url(item['url'])

    label_name = 'Melanoma' if item['label'] == 1 else 'Non-Melanoma'

    print(f"  {item['name']} ({label_name}) loaded")
    print(f"    URL: {item['url']}")

print(f'Done: {len(IMAGE_LIST)} images ready.')

Results will be saved to: /content/results
Model already present.
Model loaded. Input shape: (None, 128, 128, 3)
Reading GitHub repository...
GitHub branch found: main
Loading images...
  ISIC_0024452 (Non-Melanoma) loaded
    URL: https://raw.githubusercontent.com/Giuliana87/Samo2026/main/ISIC_0024452.jpg
  ISIC_0024658 (Non-Melanoma) loaded
    URL: https://raw.githubusercontent.com/Giuliana87/Samo2026/main/ISIC_0024658.jpg
  ISIC_0024674 (Non-Melanoma) loaded
    URL: https://raw.githubusercontent.com/Giuliana87/Samo2026/main/ISIC_0024674.jpg
  ISIC_0024710 (Non-Melanoma) loaded
    URL: https://raw.githubusercontent.com/Giuliana87/Samo2026/main/ISIC_0024710.jpg
  ISIC_0024855 (Non-Melanoma) loaded
    URL: https://raw.githubusercontent.com/Giuliana87/Samo2026/main/ISIC_0024855.jpg
  ISIC_0024940 (Melanoma) loaded
    URL: https://raw.githubusercontent.com/Giuliana87/Samo2026/main/ISIC_0024940.jpg
  ISIC_0025081 (Melanoma) loaded
    URL: https://raw.githubusercontent.com/Giuliana87

## 3. Perturbation function
Applies photometric perturbations following the paper pipeline: brightness → contrast → sharpness → saturation → hue (HSV shift).

In [6]:
def apply_perturbation(pil_img, brightness, contrast, sharpness, saturation, hue):
    img = pil_img.copy()
    img = ImageEnhance.Brightness(img).enhance(brightness)
    img = ImageEnhance.Contrast(img).enhance(contrast)
    img = ImageEnhance.Sharpness(img).enhance(sharpness)
    img = ImageEnhance.Color(img).enhance(saturation)
    # Hue shift in HSV space
    arr = np.array(img, dtype=np.uint8)
    hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:,:,0] = (hsv[:,:,0] + hue * 255) % 256
    img = Image.fromarray(cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB))
    return img

def predict_single(pil_img, model):
    x = np.expand_dims(np.array(pil_img, dtype=np.float32), axis=0)
    return float(model.predict(x, verbose=0)[0][0])

# Quick test
test_prob = predict_single(IMAGE_LIST[0]['pil'], melanoma_model)
print(f"Baseline P(melanoma) for {IMAGE_LIST[0]['name']} = {test_prob:.4f}")

Baseline P(melanoma) for ISIC_0024452 = 0.3828


## 4. Define GSA problem
Bounds from Table 1 of the paper — calibrated on dermoscopy data augmentation literature.

In [7]:
problem = {
    'num_vars': 5,
    'names': ['brightness', 'contrast', 'sharpness', 'saturation', 'hue'],
    'bounds': [
        [0.5, 1.5],    # brightness: ±50% — smartphone sensor variability
        [0.8, 1.2],    # contrast:   ±20% — typical clinical range
        [0.8, 1.2],    # sharpness:  ±20% — focus/distance variability
        [0.5, 1.5],    # saturation: ±50% — color profile differences
        [-0.1, 0.1],   # hue:        ±10% normalized — light temperature
    ]
}

print('GSA problem:')
for name, bounds in zip(problem['names'], problem['bounds']):
    print(f'  {name:12s}: {bounds}')

GSA problem:
  brightness  : [0.5, 1.5]
  contrast    : [0.8, 1.2]
  sharpness   : [0.8, 1.2]
  saturation  : [0.5, 1.5]
  hue         : [-0.1, 0.1]


## 5. Saltelli sampling
N=64 base samples → N×(D+2) = 64×7 = **448 evaluations per image**.
*(Paper uses N=256 → 1,792 per image × 935 images = 1,675,520 total inferences.)*

In [8]:
N = 64
param_values = sobol_sample.sample(problem, N=N, calc_second_order=False, seed=42)
print(f'Design matrix shape: {param_values.shape}')
print(f'Total evaluations per image: {param_values.shape[0]}')

Design matrix shape: (448, 5)
Total evaluations per image: 448


## 6. Evaluate Y
For each row of the design matrix: apply perturbation → get P(melanoma).
Change `IMAGE_IDX` to run on different images (0–4 = Melanoma, 5–9 = Non-Melanoma).

In [9]:
IMAGE_IDX = 0  # 0-4 Melanoma, 5-9 Non-Melanoma

item    = IMAGE_LIST[IMAGE_IDX]
pil_img = item['pil']
print(f"Image: {item['name']} ({'Melanoma' if item['label']==1 else 'Non-Melanoma'})")
print(f"Baseline P(melanoma) = {predict_single(pil_img, melanoma_model):.4f}")
print(f"Running {len(param_values)} model evaluations...")

Y = []
t0 = time.time()
for row in tqdm(param_values):
    perturbed = apply_perturbation(pil_img, row[0], row[1], row[2], row[3], row[4])
    Y.append(predict_single(perturbed, melanoma_model))

Y = np.array(Y)
print(f'Done in {time.time()-t0:.1f}s')
print(f'Y — mean: {Y.mean():.4f}, std: {Y.std():.4f}, min: {Y.min():.4f}, max: {Y.max():.4f}')
# Discussion:
# 1. What does Y.mean() tell you about this image?
# 2. What does Y.std() tell you? A high std means the model is sensitive to perturbations.
# 3. Try IMAGE_IDX = 5 (Non-Melanoma). How does the baseline P(melanoma) change?


Image: ISIC_0024452 (Non-Melanoma)
Baseline P(melanoma) = 0.3828
Running 448 model evaluations...


100%|██████████| 448/448 [00:37<00:00, 11.87it/s]

Done in 37.7s
Y — mean: 0.3062, std: 0.0827, min: 0.1528, max: 0.4920


## 7. Compute Sobol indices

## 7. Compute Sobol indices

**Your task:** use `sobol_analyze.analyze()` to compute S1 and ST from `Y`.

Build a DataFrame with columns `Parameter`, `S1`, `ST`, `ST-S1` and sort by ST descending.

```python
Si = sobol_analyze.analyze(problem, Y, calc_second_order=False, print_to_console=False)
```

In [ ]:
# Your code here

# Discussion:
# 1. Which factor has the highest ST? Does it match your visual intuition from Cell 8?
# 2. For which factors is ST >> S1? What does this mean in terms of interactions?
# 3. Compare with paper Table 2. The paper aggregates over 935 images — why might
#    your single-image results differ?


## 8. Visualize and compare with paper (Table 2)

In [10]:
params  = results['Parameter'].values
S1_vals = results['S1'].values
ST_vals = results['ST'].values
x = np.arange(len(params))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: your results
ax = axes[0]
ax.bar(x - w/2, S1_vals, w, label='S1', color='#4472C4', alpha=0.85)
ax.bar(x + w/2, ST_vals, w, label='ST', color='#E8611A', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(params, fontsize=11)
ax.set_ylabel('Sobol index'); ax.set_ylim(0, 1.0)
ax.set_title(f"Your results — {item['name']} (N=64)", fontsize=12)
ax.legend()

# Right: paper Table 2
paper_params = ['Hue', 'Brightness', 'Saturation', 'Contrast', 'Sharpness']
paper_S1 = [0.381, 0.242, 0.113, 0.065, 0.009]
paper_ST = [0.524, 0.375, 0.195, 0.110, 0.022]
xp = np.arange(len(paper_params))
ax2 = axes[1]
ax2.bar(xp - w/2, paper_S1, w, label='S1', color='#4472C4', alpha=0.85)
ax2.bar(xp + w/2, paper_ST, w, label='ST', color='#E8611A', alpha=0.85)
ax2.set_xticks(xp); ax2.set_xticklabels(paper_params, fontsize=11)
ax2.set_ylabel('Sobol index'); ax2.set_ylim(0, 1.0)
ax2.set_title('Paper Table 2 — aggregated over 935 images (N=256)', fontsize=12)
ax2.legend()

plt.suptitle('GSA results comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{RESULTS_PATH}/sobol_{item['name']}.png", dpi=150, bbox_inches='tight')
plt.show()

print('Does the ranking match? Why might single-image results differ from the aggregated paper values?')
# Discussion:
# 1. Does the ranking from your single image match the paper's aggregated ranking?
# 2. Why is ST for hue so much higher here than in the paper (0.524 aggregated)?
#    Hint: think about what aggregation over 935 images does to individual variance.
# 3. For which parameters is ST - S1 largest? What does this tell you about interactions?


NameError: name 'results' is not defined

## 9. Saltelli (2002) Property 1 — bounds matter

**Your task — Part A:** halve the hue bounds from `[-0.1, 0.1]` to `[-0.05, 0.05]`. Re-run and compare ST.

**Your task — Part B:** now change ALL bounds — make everything ±20% (equal ranges for all factors):
```
brightness: [0.8, 1.2]
contrast:   [0.8, 1.2]
sharpness:  [0.8, 1.2]
saturation: [0.8, 1.2]
hue:        [-0.2, 0.2]
```

Compare the ST ranking with the original. Does hue still dominate? Does the ranking change?

This is exactly the mistake Saltelli (2002) warns against: treating all factors equally.

## 9. Saltelli (2002) Property 1 — bounds matter

**Your task — Part A:** halve the hue bounds from `[-0.1, 0.1]` to `[-0.05, 0.05]`. Re-run and compare ST.

**Your task — Part B:** now change ALL bounds — make everything ±20% (equal ranges for all factors):
```
brightness: [0.8, 1.2]
contrast:   [0.8, 1.2]
sharpness:  [0.8, 1.2]
saturation: [0.8, 1.2]
hue:        [-0.2, 0.2]
```

Compare the ST ranking with the original. Does hue still dominate? Does the ranking change?

This is exactly the mistake Saltelli (2002) warns against: treating all factors equally.

In [ ]:
# Your code here

# Discussion:
# 1. Part A: when you halve hue bounds, does ST for hue go up or down? Why?
# 2. Part B: when all bounds are equal (all ±20%), does the ranking change?
#    Which factor becomes relatively more important?
# 3. What does this tell you about the paper's choice of different bounds per factor?
# 4. If you were a clinician choosing a dermoscope, which parameter would you try to control?


## 10. DenseNet-121 vs ResNet50

Do different models show the same chromatic fragility?

ResNet50 is pre-trained on ImageNet (1000 classes, not melanoma). We define **Y = 1 - P(top-1 class)**: instability of the top prediction under perturbation.

Same image, same design matrix, different model.

## 10. DenseNet-121 vs ResNet50

**Your task:** load ResNet50 pre-trained on ImageNet and run the same GSA pipeline.

Define Y = 1 - P(top-1 class) — instability of the top prediction under perturbation.

```python
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
resnet = ResNet50(weights='imagenet', include_top=True)
```

Use the same image and same `param_values` as Cell 6.

Compare the ST ranking with DenseNet-121. Is hue still dominant?

**Discussion:** is chromatic fragility a property of the model or of the image domain?

In [ ]:
# Your code here

# Discussion:
# 1. Does hue still dominate in ResNet50? Is the ranking identical to DenseNet-121?
# 2. ResNet50 was trained on ImageNet (cats, dogs, cars...), not melanoma.
#    Why might it still show chromatic fragility on a dermoscopic image?
# 3. Does this suggest that chromatic fragility is a property of the model architecture,
#    the training data, or the image domain itself?
# 4. What would you conclude about the generalizability of the paper's findings?


## 11. Final challenge — loop over multiple images

**Your task:** replicate the aggregation strategy from the paper.

Run the full GSA pipeline on all 10 images and compute the mean ST over the test set.

The paper aggregates as:
$$\bar{S} = \frac{1}{M} \sum_{i=1}^{M} S_i$$

where M is the number of images.

Suggested structure:
```python
all_ST = []  # collect ST for each image
for item in IMAGE_LIST:
    # 1. run the Y loop on item['pil']
    # 2. compute Si
    # 3. append Si['ST'] to all_ST

# aggregate
mean_ST = np.mean(all_ST, axis=0)
```

Then build a DataFrame and compare with paper Table 2.

In [ ]:
# Your code here

# Discussion:
# 1. Does the mean ST ranking over 10 images match the paper's ranking over 935?
# 2. How much does ST vary across images? Look at the standard deviation.
#    Is the result stable or image-dependent?
# 3. Is there a difference between the 5 melanoma images and the 5 non-melanoma images?
#    Compute mean ST separately for the two groups.
# 4. The paper uses N=256 and 935 images. You used N=64 and 10 images.
#    What are the trade-offs? When would you need more images vs larger N?


---
## Summary

You have replicated the core GSA pipeline from the paper:

1. **Saltelli sampling** on 5 photometric factors (N=64 → 448 evaluations)
2. **Black-box evaluation** of DenseNet-121 on perturbed images
3. **Sobol indices** via SALib — S1 and ST
4. **Bounds sensitivity** — Property 1 of Saltelli (2002)
5. **Cross-model comparison** — DenseNet-121 vs ResNet50

The paper runs this on **935 images** with N=256 → **1,675,520 total inferences**.
You ran it on 1 image with N=64 → **448 inferences**. Same framework. Same conclusions.

> Vannucci, Coppolecchia & Siciliano (2026). *Global Sensitivity Analysis for Robust XAI: Quantifying Clinical Risk and Prediction Instability in Dermoscopic Image Classification*. **Risk Analysis**.